# 01 · Train the player / ball detector on several datasets, cross-validated

The 2D pipeline needs, in every frame, every player on the court and the ball. This notebook
trains one YOLO detector for that from **several datasets at once** and scores it **on each
dataset separately**, so you can see where it works and where it doesn't.

What changed from the single-dataset version:

* **Sources, not one dataset.** Your Roboflow set, SportsMOT basketball, any other Roboflow
  Universe sets, your own labelled broadcast frames (notebook 03) and hard negatives are each a
  *source*. Every image keeps its source and its *group* (the video or sequence it came from).
* **Splits by video, per source.** Neighbouring frames of one video, and Roboflow's augmented
  copies of one image, never land on both sides of a split. The old random split let validation
  score frames the model had nearly seen, which is part of why 0.94 validation recall turned into
  6 of 10 players on a real clip.
* **Cross-dataset validation.** *Leave-one-dataset-out*: train on all sources but one, test on the
  one left out. A detector that holds up on a dataset it never saw will hold up on a new broadcast.
  Optional grouped k-fold within the pooled data as well.
* **Scores that match the pipeline**, per source: per-frame player recall (players found / players
  labelled), share of frames with every player found, ball precision / recall, phantom-ball frames,
  next to Ultralytics' mAP. Tracking (MOTA, IDF1) on held-out SportsMOT sequences.
* **New vs current model, side by side**, so `player_detector.pt` is only replaced when the new one
  is better on the footage that matters (your own frames, then SportsMOT).

Classes: `Player`, `Ref`, `Ball`, `Hoop` (plus `Clock`, `Scoreboard`, `Overlay`, which the pipeline
ignores but which teach the model what isn't a player).

In [ ]:
# Runtime > Change runtime type > GPU before running (A100 or L4 if you have them; T4 works, slower).
!nvidia-smi -L
from google.colab import drive
drive.mount("/content/drive")

# Everything that must survive a Colab disconnect (runs, weights, reports) lives in Drive.
DRIVE_ROOT = "/content/drive/MyDrive/courtvision"
import os
for d in ["models", "runs", "reports", "training_data"]:
    os.makedirs(f"{DRIVE_ROOT}/{d}", exist_ok=True)

In [ ]:
REPO_URL = "https://github.com/moorexavierb/CourtVision.git"
BRANCH = "main"   # set to a PR branch to test it, then Runtime > Restart session and run all
!rm -rf /content/courtvision && git clone -q -b {BRANCH} {REPO_URL} /content/courtvision
!pip -q install -e "/content/courtvision[models,train]"

## 1 · Sources

| Source | What it adds | Role |
|---|---|---|
| `roboflow` | your v17 set: every class, mostly full-court frames | train |
| `sportsmot` | broadcast basketball clips with every player boxed: crowds, overlaps, blur | train |
| `sportsmot_val` | SportsMOT's own val sequences, never trained on; also used for tracking scores | test |
| `own` | your NBA broadcast frames from notebook 03, corrected in Roboflow | train (most valuable) |
| `own_test` | frames from clips you keep back to score on (never trained on) | test |
| `extra_*` | other Roboflow Universe basketball sets (look through them first) | train |
| `negatives` | crowd, bench, replays, ads: nothing to label | train only |

Anything missing is skipped. Store the Roboflow key in Colab **Secrets** as `ROBOFLOW_API_KEY`
and the Hugging Face token as `HF_TOKEN`; never paste keys into cells.

In [ ]:
from google.colab import userdata
ROBOFLOW_API_KEY = userdata.get("ROBOFLOW_API_KEY")
from roboflow import Roboflow
rf = Roboflow(api_key=ROBOFLOW_API_KEY)
base = rf.workspace("workspace-5ujvu").project("basketball-players-fy4c2-vfsuv").version(17).download(
    "yolov8", location="/content/datasets/players-v17")
import yaml
NAMES = yaml.safe_load(open(f"{base.location}/data.yaml"))["names"]
NAMES = [NAMES[k] for k in sorted(NAMES)] if isinstance(NAMES, dict) else list(NAMES)
print("classes:", NAMES)

# Other Roboflow Universe sets: (workspace, project, version, {their class: our class, or None to drop}).
# Check their frames look like broadcast footage and that EVERY player is boxed: an unboxed player
# teaches the model that players are background.
# Add a set only after the vetting cell below shows it is broadcast footage with every player boxed
# and few near-duplicates of v17. Version None = latest.
EXTRA_ROBOFLOW = [
    # ("deeplearning-gmyc6", "basketball-game-detection", None, {"ball": "Ball", "player": "Player", "basket": "Hoop", "referee": "Ref"}),
    # ("basketball-w376e", "basketball-broadcast", None, {"Player": "Player", "Referee": "Ref", "Ball": "Ball", "Hoop": "Hoop"}),
]
extra_roots = []
def latest_version(project):
    return max(int(str(v.version).split("/")[-1]) for v in project.versions())

for ws, proj, ver, rename in EXTRA_ROBOFLOW:
    p = rf.workspace(ws).project(proj)
    ver = ver or latest_version(p)
    d = p.version(ver).download("yolov8", location=f"/content/datasets/{proj}-v{ver}")
    extra_roots.append((proj, d.location, rename))

### Vet candidate Roboflow Universe sets

Downloads each candidate and shows what it really contains: classes, images and boxes, image sizes
(a set exported at 640 px has balls of a few pixels), how many images are near-duplicates of your
v17 set (forks of `basketball-players-fy4c2` are the same frames re-exported, and would put copies
of your test images into training), and a grid of labelled samples to check that the footage is
broadcast and that every player is boxed. Nothing here is added to training; copy the ones that
pass into `EXTRA_ROBOFLOW` above and re-run that cell.

Candidates found on Universe (sizes from their pages where shown; check them here):

| Set | Classes | Notes |
|---|---|---|
| `deeplearning-gmyc6/basketball-game-detection` | ball, player, basket, referee | 637 images, NBA highlight frames (Warriors games 2022-24), CC BY 4.0. Best candidate |
| `basketball-w376e/basketball-broadcast` | Player, Referee, Ball, Hoop, Backboard, court regions | CC BY 4.0; broadcast by name; the court-region classes may help notebook 02 later |
| `computer-vision-d5fjh/basketball-detection-dn6fg` | ball, basket, person | CC BY 4.0; `person` may include bench and crowd, so check the grid |
| `computer-vision-and-deep-learning/nba-players-detection` | players | 71 images |
| `zeeshan-public-projects/basket-ball-tracking-xkyu5` | ? | no details on its page |
| `*/basketball-players-fy4c2*` (many forks) | same as v17 | skip: same frames as your set |

In [ ]:
import cv2, pandas as pd
from pathlib import Path
import random, numpy as np, matplotlib.pyplot as plt
from courtvision.training.datasets import _images, _names, _label_for, SPLITS
from courtvision.training.sources import near_duplicate_share

CANDIDATES = [
    ("deeplearning-gmyc6", "basketball-game-detection"),
    ("basketball-w376e", "basketball-broadcast"),
    ("computer-vision-d5fjh", "basketball-detection-dn6fg"),
    ("computer-vision-and-deep-learning", "nba-players-detection"),
    ("zeeshan-public-projects", "basket-ball-tracking-xkyu5"),
]
v17_imgs = [str(p) for s in SPLITS if os.path.isdir(f"{base.location}/{s}/images") for p in _images(Path(f"{base.location}/{s}/images"))]
v17_sample = random.Random(0).sample(v17_imgs, min(600, len(v17_imgs)))
vet = {}
for ws, proj in CANDIDATES:
    try:
        p = rf.workspace(ws).project(proj)
        ver = latest_version(p)
        root = p.version(ver).download("yolov8", location=f"/content/datasets/vet/{proj}-v{ver}").location
    except Exception as e:
        print(f"{proj}: could not download ({e})")
        continue
    names = _names(Path(root) / "data.yaml")
    imgs = [p for s in SPLITS if (Path(root) / s / "images").exists() for p in _images(Path(root) / s / "images")]
    boxes = {n: 0 for n in names}
    for im in imgs:
        lbl = _label_for(im)
        if lbl.exists():
            for line in open(lbl):
                if line.split():
                    boxes[names[int(line.split()[0])]] += 1
    sizes = [cv2.imread(str(im)).shape[:2] for im in imgs[:50]]
    dup, _ = near_duplicate_share([str(i) for i in random.Random(0).sample(imgs, min(200, len(imgs)))], v17_sample)
    vet[proj] = {"version": ver, "images": len(imgs), "boxes": boxes,
                 "typical_size": max(set(sizes), key=sizes.count) if sizes else None,
                 "near_dup_of_v17": round(dup, 2)}
    print(proj, vet[proj])
    fig, axes = plt.subplots(3, 3, figsize=(15, 9))
    for ax, im in zip(axes.flat, random.Random(1).sample(imgs, min(9, len(imgs)))):
        img = cv2.cvtColor(cv2.imread(str(im)), cv2.COLOR_BGR2RGB)
        H, W = img.shape[:2]
        if _label_for(im).exists():
            for line in open(_label_for(im)):
                c, cx, cy, w, h = line.split()[:5]
                cx, cy, w, h = float(cx) * W, float(cy) * H, float(w) * W, float(h) * H
                cv2.rectangle(img, (int(cx - w / 2), int(cy - h / 2)), (int(cx + w / 2), int(cy + h / 2)),
                              (255, 60, 60) if "ball" in names[int(c)].lower() else (60, 255, 60), 2)
                cv2.putText(img, names[int(c)], (int(cx - w / 2), int(cy - h / 2) - 3), 0, 0.5, (255, 255, 0), 1)
        ax.imshow(img); ax.axis("off")
    fig.suptitle(f"{ws}/{proj} v{vet[proj]['version']}")
    plt.tight_layout(); fig.savefig(f"{DRIVE_ROOT}/reports/vet_{proj}.jpg"); plt.show()
pd.DataFrame(vet).T if vet else None

### SportsMOT basketball

Download once (after Hugging Face approves your access request) to
`MyDrive/courtvision/training_data/sportsmot`; extract any archives so that
`train/<sequence>/img1`, `train/<sequence>/gt/gt.txt` and `splits_txt/basketball.txt` exist.

SportsMOT labels only players. The ball, referees, hoop and graphics in its frames are filled in
from your current detector's confident predictions, and frames where it is unsure about one of
them are skipped, so they are not learned as background. One frame in `SPORTSMOT_STRIDE` is used
(neighbouring frames are near duplicates). Train sequences become the `sportsmot` source; val
sequences become `sportsmot_val`, which is never trained on.

In [ ]:
SPORTSMOT_ROOT = f"{DRIVE_ROOT}/training_data/sportsmot"
DOWNLOAD_SPORTSMOT = False   # True once access is granted: downloads to Drive (several GB, once)
if DOWNLOAD_SPORTSMOT and not os.path.isdir(f"{SPORTSMOT_ROOT}/train"):
    from huggingface_hub import snapshot_download
    snapshot_download("MCG-NJU/SportsMOT", repo_type="dataset", token=userdata.get("HF_TOKEN"),
                      local_dir=SPORTSMOT_ROOT)
    print(os.listdir(SPORTSMOT_ROOT))  # extract any .zip / .tar here if the folders aren't there yet

In [ ]:
from pathlib import Path
import cv2
from ultralytics import YOLO
from courtvision.training.datasets import write_data_yaml
from courtvision.training.mining import yolo_rows
from courtvision.training.sportsmot import basketball_sequences, merge_gt_and_pseudo, read_gt

SPORTSMOT_STRIDE = 10          # 2.5 frames per second at 25 fps
SPORTSMOT_YOLO = {}
if os.path.isdir(f"{SPORTSMOT_ROOT}/train"):
    teacher = YOLO(f"{DRIVE_ROOT}/models/player_detector.pt")
    tid = {n: i for i, n in teacher.names.items()}
    fill = [i for n, i in tid.items() if n != "Player"]
    for split, name in [("train", "sportsmot"), ("val", "sportsmot_val")]:
        out = Path(f"/content/datasets/{name}-yolo")
        (out / "train" / "images").mkdir(parents=True, exist_ok=True)
        (out / "train" / "labels").mkdir(parents=True, exist_ok=True)
        kept = skipped = 0
        for seq in basketball_sequences(SPORTSMOT_ROOT, split):
            gt = read_gt(seq)
            frames = {int(p.stem): p for p in (seq / "img1").glob("*.jpg")}
            for f in sorted(gt)[::SPORTSMOT_STRIDE]:
                img = cv2.imread(str(frames[f])) if f in frames else None
                if img is None:
                    continue
                b = teacher.predict(img, imgsz=1280, conf=0.15, verbose=False)[0].boxes
                merged = merge_gt_and_pseudo(gt[f], b.xyxy.cpu().numpy(), b.cls.cpu().numpy(), b.conf.cpu().numpy(),
                                             player_cls=tid["Player"], fill_classes=fill)
                if merged is None:
                    skipped += 1
                    continue
                stem = f"{seq.name}_{f:06d}"     # sequence name first: frames of a sequence share a group
                cv2.imwrite(str(out / "train" / "images" / f"{stem}.jpg"), img)
                (out / "train" / "labels" / f"{stem}.txt").write_text("\n".join(yolo_rows(*merged, (img.shape[1], img.shape[0]))) + "\n")
                kept += 1
        write_data_yaml(out, [teacher.names[i] for i in sorted(teacher.names)])
        SPORTSMOT_YOLO[name] = str(out)
        print(f"{name}: {kept} frames kept, {skipped} skipped as ambiguous")

## 2 · Build the combined dataset

`weight` repeats a source's training images: give your own broadcast frames 2 to 3 so a few
hundred of them are not drowned out by thousands of SportsMOT frames. Validation and test are
never repeated. The table shows images and boxes per class for each source and split; a class with
fewer than ~100 validation boxes in a source gives a noisy score for that source.

In [ ]:
from courtvision.training.sources import Source, build_multisource, summary

TD = f"{DRIVE_ROOT}/training_data"
SOURCES = [Source("roboflow", base.location)]
SOURCES += [Source(f"extra_{p.replace('-', '_')}", root, rename=r) for p, root, r in extra_roots]
if "sportsmot" in SPORTSMOT_YOLO:
    SOURCES.append(Source("sportsmot", SPORTSMOT_YOLO["sportsmot"], group_by="video"))
if "sportsmot_val" in SPORTSMOT_YOLO:
    SOURCES.append(Source("sportsmot_val", SPORTSMOT_YOLO["sportsmot_val"], role="test", group_by="video"))
if os.path.exists(f"{TD}/own_labelled/data.yaml"):
    SOURCES.append(Source("own", f"{TD}/own_labelled", weight=3))
if os.path.exists(f"{TD}/own_test/data.yaml"):
    SOURCES.append(Source("own_test", f"{TD}/own_test", role="test"))
if os.path.isdir(f"{TD}/negatives"):
    SOURCES.append(Source("negatives", f"{TD}/negatives", role="negatives"))

MULTI = "/content/datasets/multi"
manifest = build_multisource(SOURCES, NAMES, MULTI, val_frac=0.15, test_frac=0.15, seed=0)
DATA_YAML = f"{MULTI}/data.yaml"
print("groups per source:", manifest.groupby("source").group.nunique().to_dict())
display(summary(manifest, NAMES))
manifest.to_csv(f"{DRIVE_ROOT}/reports/manifest.csv", index=False)

## 3 · Cross-dataset validation (leave one dataset out)

For each training source: train on the others, test on every image of the one left out. Short
runs on a small model, because the point is the *comparison between datasets*, not the best
weights. Read the table like this:

* a source with **low held-out recall** looks different from the rest: the model needs its data
  (and the real pipeline footage probably looks more like it than you think if it's broadcast);
* if **SportsMOT held out** scores well, the Roboflow data already generalises to broadcast
  crowds; if it scores badly, SportsMOT is filling a real gap.

About 15 to 25 minutes per fold on an A100 at the defaults. Set `RUN_LODO = False` to skip.

In [ ]:
import pandas as pd
from courtvision.training.evaluate import score_images, summarise
from courtvision.training.sources import lodo_folds

RUN_LODO = True
CV_MODEL, CV_IMGSZ, CV_EPOCHS = "yolo11s.pt", 960, 30
# Thresholds the pipeline actually runs at (configs/default.yaml: detection.conf, detection.ball_conf).
CONF = {"Player": 0.25, "Ball": 0.15, "Ref": 0.25}

def evaluate(weights, data_yaml, images, imgsz=1280):
    """mAP from Ultralytics plus the pipeline's per-frame scores, on the yaml's test split."""
    m = YOLO(weights)
    row = {}
    try:
        val = m.val(data=data_yaml, imgsz=imgsz, split="test", verbose=False, plots=False)
        row["mAP50"] = round(float(val.box.map50), 3)
        present = list(val.box.ap_class_index)
        for n in ("Player", "Ball"):
            if NAMES.index(n) in present:
                row[f"{n}_mAP50"] = round(float(val.box.class_result(present.index(NAMES.index(n)))[2]), 3)
    except Exception as e:  # e.g. no labels of any class in the split
        print("val skipped:", e)
    row.update(summarise(score_images(m, images, NAMES, CONF, imgsz=imgsz)))
    return row

lodo = {}
if RUN_LODO:
    for held, fold_yaml in lodo_folds(MULTI, NAMES).items():
        name = f"lodo_{held}_{CV_MODEL.split('.')[0]}_{CV_IMGSZ}"
        run = f"{DRIVE_ROOT}/runs/cv/{name}"
        if not os.path.exists(f"{run}/weights/best.pt"):
            YOLO(CV_MODEL).train(data=str(fold_yaml), imgsz=CV_IMGSZ, epochs=CV_EPOCHS, patience=10, batch=-1,
                                 cos_lr=True, close_mosaic=5, project=f"{DRIVE_ROOT}/runs/cv", name=name,
                                 exist_ok=True, plots=False)
        test_imgs = open(yaml.safe_load(open(fold_yaml))["test"]).read().split()
        lodo[held] = evaluate(f"{run}/weights/best.pt", str(fold_yaml), test_imgs, CV_IMGSZ)
        print(held, lodo[held])
    lodo_df = pd.DataFrame(lodo).T
    lodo_df.to_csv(f"{DRIVE_ROOT}/reports/lodo.csv")
    display(lodo_df[[c for c in ["frames", "player_recall_per_frame", "all_players_found_share", "Player_precision",
                                 "Ball_recall", "Ball_precision", "phantom_ball_frame_share", "mAP50"] if c in lodo_df]])

### Optional: grouped k-fold

Classic cross-validation over the pooled training data, with whole videos per fold. The spread
across folds tells you how much a single validation score can be trusted. Off by default (it is
k more training runs).

In [ ]:
from courtvision.training.sources import kfold_folds
RUN_KFOLD, K = False, 3
if RUN_KFOLD:
    kf = {}
    for i, fold_yaml in enumerate(kfold_folds(MULTI, NAMES, k=K)):
        name = f"kfold{i}_{CV_MODEL.split('.')[0]}_{CV_IMGSZ}"
        YOLO(CV_MODEL).train(data=str(fold_yaml), imgsz=CV_IMGSZ, epochs=CV_EPOCHS, patience=10, batch=-1,
                             project=f"{DRIVE_ROOT}/runs/cv", name=name, exist_ok=True, plots=False)
        val_imgs = open(yaml.safe_load(open(fold_yaml))["val"]).read().split()
        kf[i] = summarise(score_images(YOLO(f"{DRIVE_ROOT}/runs/cv/{name}/weights/best.pt"), val_imgs, NAMES, CONF,
                                       imgsz=CV_IMGSZ))
    kf_df = pd.DataFrame(kf).T
    display(kf_df.describe().loc[["mean", "std"]])

## 4 · Train on everything

Broadcast frames are wide and the ball is ~15-25 px, so train at `imgsz=1280` and keep mosaic on.
`fliplr` is fine (left/right doesn't change a class). Runs save to Drive every 10 epochs and resume
after a disconnect. Use a new `RUN_NAME` whenever the data changes, or a finished run would be
"resumed" and stop at once.

In [ ]:
RUN_NAME = "detector_multi_yolo11m_1280_v1"
last = f"{DRIVE_ROOT}/runs/{RUN_NAME}/weights/last.pt"
resume = os.path.exists(last)
YOLO(last if resume else "yolo11m.pt").train(
    data=DATA_YAML, imgsz=1280, epochs=150, patience=30, batch=-1,
    cos_lr=True, close_mosaic=15, save_period=10,
    project=f"{DRIVE_ROOT}/runs", name=RUN_NAME, exist_ok=True, resume=resume,
)
NEW = f"{DRIVE_ROOT}/runs/{RUN_NAME}/weights/best.pt"

## 5 · Score per source: new model vs current model

Each source's **test** split (whole videos neither model trained on in this build; the current
model may have seen Roboflow test images in its own training, so its `roboflow` row flatters it).
The rows that matter most are `own_test`, then `own`, then `sportsmot_val`.

In [ ]:
CURRENT = f"{DRIVE_ROOT}/models/player_detector.pt"
rows = []
for src in sorted(manifest.source.unique()):
    if src == "negatives":
        continue
    imgs = manifest[(manifest.source == src) & (manifest.split == "test")].image.tolist()
    if not imgs:
        continue
    for label, w in [("new", NEW), ("current", CURRENT)]:
        if os.path.exists(w):
            rows.append({"source": src, "model": label, **evaluate(w, f"{MULTI}/data_{src}.yaml", imgs)})
per_source = pd.DataFrame(rows).set_index(["source", "model"])
per_source.to_csv(f"{DRIVE_ROOT}/reports/per_source_{RUN_NAME}.csv")
display(per_source[[c for c in ["frames", "player_recall_per_frame", "all_players_found_share",
                                "players_found_per_frame", "players_labelled_per_frame", "Player_precision",
                                "Ball_recall", "Ball_precision", "phantom_ball_frame_share", "mAP50"]
                    if c in per_source]])

## 6 · Tracking on held-out SportsMOT sequences

The 2D output is only useful if each court position stays attached to one player, so the
detector is also scored *through the tracker the pipeline uses* (ByteTrack, same settings as
`configs/default.yaml`) on SportsMOT's val sequences: **MOTA** (misses, false boxes and identity
switches against all labelled boxes) and **IDF1** (how much of each player's time is kept under one
id). SportsMOT boxes only players, so referees are left out on both sides.

In [ ]:
from courtvision.config import Config
from courtvision.detection.tracker import ObjectTracker
from courtvision.training.evaluate import mot_metrics, mot_table
from courtvision.training.sportsmot import read_gt_tracks, seq_fps

MAX_SEQS = 8   # each sequence is a few hundred frames
def track_scores(weights):
    cfg = Config.load("/content/courtvision/configs/default.yaml")
    out = {}
    for seq in basketball_sequences(SPORTSMOT_ROOT, "val")[:MAX_SEQS]:
        imgs = sorted((seq / "img1").glob("*.jpg"))
        trk = ObjectTracker(weights, cfg.detection, fps=seq_fps(seq))
        det = trk.run((cv2.imread(str(p)) for p in imgs), total=len(imgs))
        pred = {int(p.stem): {t: d["bbox"] for t, d in fr.items()} for p, fr in zip(imgs, det.players)}
        out[seq.name] = mot_metrics(read_gt_tracks(seq), pred)
    return mot_table(out)

if os.path.isdir(f"{SPORTSMOT_ROOT}/val"):
    tracking = {label: track_scores(w) for label, w in [("new", NEW), ("current", CURRENT)] if os.path.exists(w)}
    report = pd.concat({k: v.loc[["overall"]] for k, v in tracking.items()})
    report.to_csv(f"{DRIVE_ROOT}/reports/tracking_{RUN_NAME}.csv")
    display(report)

## 7 · Keep the new model if it wins

Replace `player_detector.pt` only if the new model has higher per-frame player recall on the
broadcast sources without more phantom balls. The old file is kept as `player_detector_prev.pt`.

In [ ]:
import shutil
def better(src):
    if src not in per_source.index.get_level_values(0):
        return None
    if (src, "current") not in per_source.index:
        return True
    n, c = per_source.loc[(src, "new")], per_source.loc[(src, "current")]
    return (n.get("player_recall_per_frame", 0) >= c.get("player_recall_per_frame", 0)
            and n.get("phantom_ball_frame_share", 1) <= c.get("phantom_ball_frame_share", 1) + 0.02)
verdict = {s: better(s) for s in ["own_test", "own", "sportsmot_val", "roboflow"]}
print(verdict)
decider = next((v for v in verdict.values() if v is not None), True)
if decider:
    if os.path.exists(CURRENT):
        shutil.copy(CURRENT, f"{DRIVE_ROOT}/models/player_detector_prev.pt")
    shutil.copy(NEW, CURRENT)
    print("Saved", CURRENT)
else:
    print("Kept the current model; the new one is in", NEW)

## Optional: dedicated ball model

If `Ball_recall` on the broadcast sources is below ~0.85, train a ball-only model starting from the
detector just trained (its backbone already knows broadcast basketball), on the same grouped
splits. Point `models.ball_detector` in `configs/default.yaml` at the result. It is only saved if it
beats the main detector's ball recall on the test images.

In [ ]:
BALL_ROOT = Path("/content/datasets/ball-only")
shutil.rmtree(BALL_ROOT, ignore_errors=True)
ball_id = NAMES.index("Ball")
lists = {}
for split in ["train", "valid", "test"]:
    keep = manifest[manifest.split == split]   # frames without a ball stay in: they teach "no ball here"
    paths = []
    for img in keep.image:
        rel = Path(img).relative_to(f"{MULTI}/images")
        dst = BALL_ROOT / "images" / rel
        dst.parent.mkdir(parents=True, exist_ok=True)
        if not dst.exists():
            os.symlink(img, dst)
        lbl_src = Path(MULTI) / "labels" / rel.with_suffix(".txt")
        lbl_dst = BALL_ROOT / "labels" / rel.with_suffix(".txt")
        lbl_dst.parent.mkdir(parents=True, exist_ok=True)
        rows = [" ".join(["0"] + l.split()[1:]) for l in open(lbl_src) if l.split() and int(l.split()[0]) == ball_id]
        lbl_dst.write_text("\n".join(rows) + ("\n" if rows else ""))
        paths.append(str(dst))
    (BALL_ROOT / f"{split}.txt").write_text("\n".join(paths) + "\n")
yaml.safe_dump({"path": str(BALL_ROOT), "train": str(BALL_ROOT / "train.txt"), "val": str(BALL_ROOT / "valid.txt"),
                "test": str(BALL_ROOT / "test.txt"), "names": {0: "Ball"}}, open(BALL_ROOT / "data.yaml", "w"))

BALL_RUN = "ball_multi_from_detector_1280_v1"
last = f"{DRIVE_ROOT}/runs/{BALL_RUN}/weights/last.pt"
resume = os.path.exists(last)
YOLO(last if resume else NEW).train(data=str(BALL_ROOT / "data.yaml"), imgsz=1280, epochs=150, patience=40, batch=-1,
                                    cos_lr=True, close_mosaic=20, save_period=10, lr0=0.002, warmup_epochs=1,
                                    project=f"{DRIVE_ROOT}/runs", name=BALL_RUN, exist_ok=True, resume=resume)
ball_best = YOLO(f"{DRIVE_ROOT}/runs/{BALL_RUN}/weights/best.pt")
test_imgs = (BALL_ROOT / "test.txt").read_text().split()
ball_only = summarise(score_images(ball_best, test_imgs, ["Ball"], {"Ball": CONF["Ball"]}))
main = summarise(score_images(YOLO(NEW), manifest[manifest.split == "test"].image.tolist(), NAMES, {"Ball": CONF["Ball"]}))
print("ball-only:", ball_only, "\nmain detector:", main)
if (ball_only.get("Ball_recall") or 0) > (main.get("Ball_recall") or 0):
    shutil.copy(f"{DRIVE_ROOT}/runs/{BALL_RUN}/weights/best.pt", f"{DRIVE_ROOT}/models/ball_detector.pt")
    print("Saved ball_detector.pt")